In [30]:
# !pip install PyPDF2
# !pip install tiktoken
# !pip install --upgrade google-cloud-aiplatform


In [ ]:
# !pip show google-cloud-aiplatform.

In [59]:
from google.cloud import storage
from google.cloud import aiplatform
import PyPDF2
import io
import numpy as np
import json
from typing import List, Dict, Generator, Any
import tiktoken
from dataclasses import dataclass
import logging

class PDFStreamer:
    def __init__(self, bucket_name: str):
        self.storage_client = storage.Client()
        self.bucket = self.storage_client.bucket(bucket_name)
        self.enc = tiktoken.get_encoding("cl100k_base")

    def stream_pdf(self, pdf_name: str) -> Generator[tuple[str, int, str], None, None]:
        """
        Stream PDF content page by page with metadata
        Returns: Generator of (text, page_number, section_heading)
        """
        blob = self.bucket.blob(pdf_name)
        pdf_content = io.BytesIO()
        blob.download_to_file(pdf_content)
        pdf_content.seek(0)
        
        pdf_reader = PyPDF2.PdfReader(pdf_content)
        current_section = ""
        
        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            text = page.extract_text()
            
            # Extract section heading if present
            lines = text.split('\n')
            if lines and any(char.isupper() for char in lines[0]):
                current_section = lines[0].strip()
            
            yield text, page_num + 1, current_section
            
class TextChunker:
    def __init__(self, max_tokens: int = 512, overlap_tokens: int = 50):
        self.max_tokens = max_tokens
        self.overlap_tokens = overlap_tokens
        self.enc = tiktoken.get_encoding("cl100k_base")

    def chunk_text(self, 
                  text: str, 
                  page_number: int, 
                  section_heading: str,
                  pdf_name: str) -> List[tuple[str, ChunkMetadata]]:
        """
        Chunk text by tokens with metadata
        """
        tokens = self.enc.encode(text)
        chunks = []
        start_idx = 0
        
        while start_idx < len(tokens):
            end_idx = min(start_idx + self.max_tokens, len(tokens))
            chunk_tokens = tokens[start_idx:end_idx]
            chunk_text = self.enc.decode(chunk_tokens)
            
            metadata = ChunkMetadata(
                page_number=page_number,
                section_heading=section_heading,
                start_index=start_idx,
                end_index=end_idx,
                pdf_name=pdf_name
            )
            
            chunks.append((chunk_text, metadata))
            start_idx = end_idx - self.overlap_tokens
            
        return chunks

@dataclass
class ChunkMetadata:
    page_number: int
    section_heading: str
    start_index: int
    end_index: int
    pdf_name: str

class VertexVectorIndex:
    def __init__(self, project_id: str, location: str, index_id: str, dimensions: int = 768):
        self.project_id = project_id
        self.location = location
        self.index_id = index_id
        self.dimensions = dimensions
        self.tracking_blob_name = f"processing_status/{index_id}/processed_files.json"
        aiplatform.init(project=project_id, location=location)
        self.index = self._get_or_create_index()
        
    def _get_or_create_index(self):
        """Get existing index or create a new one."""
        try:
            # Try to get existing index using fully qualified name
            index_name = f"projects/{self.project_id}/locations/{self.location}/indexes/{self.index_id}"
            index = aiplatform.MatchingEngineIndex(index_name=index_name)
            logging.info(f"Found existing index: {self.index_id}")
            return index
        except Exception as e:
            logging.info(f"Creating new index: {str(e)}")
            
            # Step 1: Create the index using tree-AH algorithm
            display_name = f"index-{self.index_id}"  # Ensure unique display name
            index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
                display_name=display_name,
                contents_delta_uri="",  # Empty string for online-only indexing
                dimensions=self.dimensions,
                approximate_neighbors_count=50,
                leaf_node_embedding_count=500,
                leaf_nodes_to_search_percent=10,
                distance_measure_type="COSINE_DISTANCE",
                description=f"Vector index for {self.index_id}",
                labels={"created_by": "vertex_vector_index"}
            )
            
            # Step 2: Wait for index creation to complete
            index.wait()
            
            # Step 3: Create an endpoint with a display name
            endpoint_display_name = f"endpoint-{self.index_id}"
            endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
                display_name=endpoint_display_name,
                description=f"Endpoint for {self.index_id}",
                # network=None  # Use default VPC
                public_endpoint_enabled=True
            )
            
            # Step 4: Wait for endpoint creation
            endpoint.wait()
            
            # Step 5: Deploy index to endpoint
            deployed_index = endpoint.deploy_index(
                index=index,
                deployed_index_id=self.index_id,
                min_replica_count=1,
                max_replica_count=2
            )
            
            logging.info(f"Created and deployed index: {self.index_id}")
            return index

    def update_index(self, vectors: List[List[float]], metadata_list: List[ChunkMetadata]):
        """Update index with new vectors and metadata."""
        datapoints = []
        for i, (vector, metadata) in enumerate(zip(vectors, metadata_list)):
            datapoint = {
                "datapoint_id": f"dp_{int(time.time())}_{i}",
                "feature_vector": vector,
                "restricts": {
                    "page_number": str(metadata.page_number),
                    "section_heading": metadata.section_heading,
                    "pdf_name": metadata.pdf_name
                }
            }
            datapoints.append(datapoint)
        
        # Update in batches
        batch_size = 100
        for i in range(0, len(datapoints), batch_size):
            batch = datapoints[i:i + batch_size]
            operation = self.index.upsert_datapoints(datapoints=batch)
            operation.result()  # Wait for batch completion
            logging.info(f"Uploaded batch {i//batch_size + 1}")

    def search_similar(self, query_vector: List[float], num_neighbors: int = 5):
        """Search for similar vectors."""
        # Get the deployed index endpoint
        endpoint_name = f"projects/{self.project_id}/locations/{self.location}/indexEndpoints/{self.index_id}-endpoint"
        try:
            endpoint = aiplatform.MatchingEngineIndexEndpoint(
                index_endpoint_name=endpoint_name
            )
            
            # Find nearest neighbors
            response = endpoint.match(
                deployed_index_id=f"{self.index_id}_deployed",
                queries=[query_vector],
                num_neighbors=num_neighbors
            )
            return response
        except Exception as e:
            logging.error(f"Error searching index: {str(e)}")
            raise
            
class ProcessingTracker:
    def __init__(self, bucket_name: str, index_id: str):
        self.storage_client = storage.Client()
        self.bucket = self.storage_client.bucket(bucket_name)
        self.index_id = index_id
        self.tracking_blob_name = f"processing_status/{index_id}/processed_files.json"

    def get_processed_files(self) -> set:
        """Retrieve list of already processed files"""
        blob = self.bucket.blob(self.tracking_blob_name)
        if not blob.exists():
            return set()
        
        try:
            content = blob.download_as_string()
            processed_files = json.loads(content)
            return set(processed_files)
        except Exception as e:
            logging.error(f"Error reading processed files: {str(e)}")
            return set()

    def mark_as_processed(self, pdf_name: str):
        """Mark a file as successfully processed"""
        processed_files = self.get_processed_files()
        processed_files.add(pdf_name)
        
        blob = self.bucket.blob(self.tracking_blob_name)
        blob.upload_from_string(
            json.dumps(list(processed_files)),
            content_type='application/json'
        )
        
class TextbookProcessor:
    def __init__(self, 
                 project_id: str,
                 bucket_name: str,
                 location: str = "us-central1",
                 index_id: str = "textbook-embeddings"):
        self.project_id = project_id
        self.bucket_name = bucket_name
        self.location = location
        self.index_id = index_id
        
        # Initialize components
        self.pdf_streamer = PDFStreamer(bucket_name)
        self.chunker = TextChunker()
        self.tracker = ProcessingTracker(bucket_name, index_id)
        self.vector_index = None  # Initialize later to handle endpoint creation

    def initialize_vector_index(self):
        """Initialize or reconnect to vector index"""
        if self.vector_index is None:
            self.vector_index = VertexVectorIndex(
                project_id=self.project_id,
                location=self.location,
                index_id=self.index_id
            )

    def process_pdf(self, pdf_name: str):
        """Process PDF with tracking"""
        if pdf_name in self.tracker.get_processed_files():
            logging.info(f"Skipping already processed file: {pdf_name}")
            return

        try:
            # Ensure vector index is initialized
            if self.vector_index is None:
                self.initialize_vector_index()

            # Initialize embedding model
            model = aiplatform.TextEmbeddingModel.from_pretrained(
                "textembedding-gecko@latest"
            )
            
            # Process PDF in streams
            for text, page_num, section in self.pdf_streamer.stream_pdf(pdf_name):
                chunks_with_metadata = self.chunker.chunk_text(
                    text, page_num, section, pdf_name
                )
                
                # Generate embeddings
                chunk_texts = [chunk[0] for chunk in chunks_with_metadata]
                embeddings = model.get_embeddings(chunk_texts)
                vectors = [emb.values for emb in embeddings]
                
                # Update vector index
                metadata_list = [chunk[1] for chunk in chunks_with_metadata]
                self.vector_index.update_index(vectors, metadata_list)

            # Mark as processed after successful completion
            self.tracker.mark_as_processed(pdf_name)
            logging.info(f"Successfully processed and marked: {pdf_name}")
            
        except Exception as e:
            logging.error(f"Error processing {pdf_name}: {str(e)}")
            raise

def process_folder(
    project_id: str,
    bucket_name: str,
    folder_prefix: str,
    location: str = "us-central1",
    index_name: str = "textbook-embeddings"
) -> None:
    """
    Process all PDF files in a GCS folder
    """
    logging.basicConfig(level=logging.INFO)
    
    # Initialize storage client
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    
    # List all PDFs in folder
    blobs = bucket.list_blobs(prefix=folder_prefix)
    pdf_files = [
        blob.name 
        for blob in blobs 
        if blob.name.lower().endswith('.pdf')
    ]
    
    if not pdf_files:
        logging.warning(f"No PDF files found in {folder_prefix}")
        return
        
    # Initialize processor
    processor = TextbookProcessor(
        project_id=project_id,
        bucket_name=bucket_name,
        location=location,
        index_name=index_name
    )
    
    # Process each PDF
    for pdf_file in pdf_files:
        try:
            logging.info(f"Processing {pdf_file}")
            processor.process_pdf(pdf_file)
            logging.info(f"Successfully processed {pdf_file}")
        except Exception as e:
            logging.error(f"Error processing {pdf_file}: {str(e)}")
            continue

def process_folder_with_batching(
    project_id: str,
    bucket_name: str,
    folder_prefix: str,
    batch_size: int = 5,
    location: str = "us-central1",
    index_name: str = "textbook-embeddings"
) -> None:
    """
    Process PDFs in batches to optimize resource usage
    """
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    
    # List and filter PDFs
    blobs = bucket.list_blobs(prefix=folder_prefix)
    pdf_files = [
        blob.name 
        for blob in blobs 
        if blob.name.lower().endswith('.pdf')
    ]
    
    # Process in batches
    for i in range(0, len(pdf_files), batch_size):
        batch = pdf_files[i:i + batch_size]
        print(index_name)
        # Process batch in parallel
        processor = TextbookProcessor(
        project_id=project_id,
        bucket_name=bucket_name,
        location=location,
        index_id=f"{index_name}-batch-{i//batch_size}"  # Changed parameter name here
    )
        
        for pdf_file in batch:
            try:
                logging.info(f"Processing {pdf_file}")
                processor.process_pdf(pdf_file)
            except Exception as e:
                logging.error(f"Error processing {pdf_file}: {str(e)}")
                continue
        
        logging.info(f"Completed batch {i//batch_size + 1}")

def list_processed_files(
    bucket_name: str,
    index_name: str
) -> tuple[List[str], List[str]]:
    """
    Compare processed files with source files
    Returns: (processed_files, unprocessed_files)
    """
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    
    # Get all PDF files
    pdf_blobs = bucket.list_blobs(prefix='pdfs/')
    pdf_files = set(
        blob.name 
        for blob in pdf_blobs 
        if blob.name.lower().endswith('.pdf')
    )
    
    # Get processed files from metadata
    processed_blobs = bucket.list_blobs(prefix=f'metadata/{index_name}')
    processed_files = set(
        blob.name.replace('metadata/', '').replace('_metadata.json', '.pdf')
        for blob in processed_blobs
    )
    
    # Find unprocessed files
    unprocessed_files = pdf_files - processed_files
    
    return list(processed_files), list(unprocessed_files)

def resume_processing(
    project_id: str,
    bucket_name: str,
    folder_prefix: str,
    location: str = "us-central1",
    index_name: str = "textbook-embeddings"
) -> None:
    """
    Resume processing from last successful file
    """
    # Get processed and unprocessed files
    processed_files, unprocessed_files = list_processed_files(
        bucket_name, index_name
    )
    
    if not unprocessed_files:
        logging.info("All files have been processed")
        return
        
    logging.info(f"Resuming processing for {len(unprocessed_files)} files")
    
    # Process remaining files
    processor = TextbookProcessor(
        project_id=project_id,
        bucket_name=bucket_name,
        location=location,
        index_name=index_name
    )
    
    for pdf_file in unprocessed_files:
        try:
            logging.info(f"Processing {pdf_file}")
            processor.process_pdf(pdf_file)
        except Exception as e:
            logging.error(f"Error processing {pdf_file}: {str(e)}")
            continue

def resume_processing(
    project_id: str,
    bucket_name: str,
    folder_prefix: str,
    location: str = "us-central1",
    index_id: str = "textbook-embeddings"
) -> None:
    """
    Resume processing with enhanced error handling and progress tracking
    """
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    
    # Initialize processor
    processor = TextbookProcessor(
        project_id=project_id,
        bucket_name=bucket_name,
        location=location,
        index_id=index_id
    )
    
    # Get all PDF files in folder
    blobs = bucket.list_blobs(prefix=folder_prefix)
    pdf_files = [
        blob.name 
        for blob in blobs 
        if blob.name.lower().endswith('.pdf')
    ]
    
    # Get already processed files
    processed_files = processor.tracker.get_processed_files()
    files_to_process = [f for f in pdf_files if f not in processed_files]
    
    logging.info(f"Found {len(files_to_process)} files to process")
    
    # Process remaining files
    for pdf_file in files_to_process:
        try:
            logging.info(f"Processing {pdf_file}")
            processor.process_pdf(pdf_file)
        except Exception as e:
            logging.error(f"Error processing {pdf_file}: {str(e)}")
            continue
        
    logging.info("Completed processing all remaining files")
                 
# Example usage
if __name__ == "__main__":
    project_id = "machine-learning-359922"
    bucket_name = "raw-files-mw"
    folder_prefix = "textbooks/"
    
    # Simple processing
    # process_folder(
    #     project_id=project_id,
    #     bucket_name=bucket_name,
    #     folder_prefix=folder_prefix
    # )

#     # Or batch processing
#     process_folder_with_batching(
#         project_id=project_id,
#         bucket_name=bucket_name,
#         folder_prefix=folder_prefix,
#         batch_size=5
#     )

# Or resume interrupted processing
resume_processing(
    project_id=project_id,
    bucket_name=bucket_name,
    folder_prefix=folder_prefix
)

INFO:root:Found 12 files to process
INFO:root:Processing textbooks/2201.02135.pdf
INFO:root:Creating new index: 400 Request contains an invalid argument.


Creating MatchingEngineIndex


INFO:google.cloud.aiplatform.matching_engine.matching_engine_index:Creating MatchingEngineIndex


Create MatchingEngineIndex backing LRO: projects/508029733371/locations/us-central1/indexes/8996182148230676480/operations/720783644997713920


INFO:google.cloud.aiplatform.matching_engine.matching_engine_index:Create MatchingEngineIndex backing LRO: projects/508029733371/locations/us-central1/indexes/8996182148230676480/operations/720783644997713920
ERROR:root:Error processing textbooks/2201.02135.pdf: 429 The following quotas are exceeded: MatchingEngineIndexes
ERROR:root:Error processing textbooks/2201.02135.pdf: 429 The following quotas are exceeded: MatchingEngineIndexes
INFO:root:Processing textbooks/2301.04856.pdf
INFO:root:Creating new index: 400 Request contains an invalid argument.


...

INFO:google.cloud.aiplatform.matching_engine.matching_engine_index:Create MatchingEngineIndex backing LRO: projects/508029733371/locations/us-central1/indexes/8847563360527450112/operations/5296018653941071872
ERROR:root:Error processing textbooks/mml-book.pdf: 429 The following quotas are exceeded: MatchingEngineIndexes
ERROR:root:Error processing textbooks/mml-book.pdf: 429 The following quotas are exceeded: MatchingEngineIndexes
INFO:root:Processing textbooks/thebook.pdf
INFO:root:Creating new index: 400 Request contains an invalid argument.


Creating MatchingEngineIndex


INFO:google.cloud.aiplatform.matching_engine.matching_engine_index:Creating MatchingEngineIndex


Create MatchingEngineIndex backing LRO: projects/508029733371/locations/us-central1/indexes/964575162768818176/operations/1594059760242524160


INFO:google.cloud.aiplatform.matching_engine.matching_engine_index:Create MatchingEngineIndex backing LRO: projects/508029733371/locations/us-central1/indexes/964575162768818176/operations/1594059760242524160
ERROR:root:Error processing textbooks/thebook.pdf: 429 The following quotas are exceeded: MatchingEngineIndexes
ERROR:root:Error processing textbooks/thebook.pdf: 429 The following quotas are exceeded: MatchingEngineIndexes
INFO:root:Completed processing all remaining files
